# T5 — the text-to-text encoder-decoder Transformer

> Tutorial pair for [`t5.py`](t5.py). Read
> [`transformer.ipynb`](transformer.ipynb) first.

## 1. Intuition
T5's thesis: **cast every NLP task as "text in -> text out."** Translation,
summarization, classification, even regression all become string-to-string
problems handled by *one* model with *one* loss. Architecturally it is the
original Transformer — a **bidirectional encoder** reads the input, a **causal
decoder** writes the output while cross-attending to the encoder. It is pretrained
by **span corruption**: drop random spans from the input and make the decoder
regenerate them.

## 2. Concept (the slide)
- **Encoder–decoder:** encoder = bidirectional self-attention; decoder = causal
  self-attention **+ cross-attention** to the encoder memory.
- **Shared vocabulary & embedding** for input and output (tied to the output
  projection).
- **Span corruption** pretraining uses **sentinel tokens** (`<extra_id_0>`, …):
  each dropped span is one sentinel in the input, and the target is the sentinels
  followed by their missing content.
- **Teacher forcing + causal mask** in the decoder, just like the base
  Transformer.

## 3. Math derivation

### 3.1 The text-to-text framing
Every task is a conditional sequence model
$$p_\theta(y_{1:T_y}\mid x_{1:T_x})=\prod_{t=1}^{T_y} p_\theta(y_t\mid y_{<t},\,x_{1:T_x}),$$
trained with the same token-level cross-entropy
$\mathcal{L}=-\sum_t \log p_\theta(y_t\mid y_{<t},x)$ regardless of whether $y$ is a
translation, a class name ("positive"), or a number rendered as text. The encoder
provides the conditioning $x$ via **cross-attention**: in the decoder,
$$\text{CrossAttn}(Q_{\text{dec}},K_{\text{enc}},V_{\text{enc}})
=\operatorname{softmax}\!\Big(\tfrac{Q_{\text{dec}}K_{\text{enc}}^\top}{\sqrt{d_k}}\Big)V_{\text{enc}},$$
where queries come from the decoder and keys/values from the encoder output.

### 3.2 Span corruption (the pretraining objective)
Sample a corruption rate $\rho$ (T5 uses $\approx 15\%$). Mark tokens i.i.d. with
probability $\rho$, then merge **consecutive** marked tokens into spans. Replace
each span $s_k$ in the input by a unique sentinel $\langle X_k\rangle$, and define
the target as the sentinels interleaved with the dropped spans:
$$
\underbrace{\text{the}\ \langle X_0\rangle\ \text{walked}\ \langle X_1\rangle\ \text{dog}}_{\text{encoder input}}
\;\longrightarrow\;
\underbrace{\langle X_0\rangle\ \text{cat}\ \langle X_1\rangle\ \text{the}\ \langle\text{EOS}\rangle}_{\text{decoder target}}.
$$
The loss is the usual decoder cross-entropy over this short target. Because the
target contains only the *missing* spans (not the whole input), each example is
cheap and dense in signal — a middle ground between BERT's MLM (predict isolated
tokens) and full sequence reconstruction.

### 3.3 Why an encoder-decoder (vs decoder-only)?
The encoder sees the source **bidirectionally** (good for understanding), while
the decoder stays autoregressive (good for generation). Cross-attention is the
bridge: every output step can query the entire, fully-contextualized source.

## 4. Key component — span corruption + the cross-attending decoder layer

In [ ]:
# ===== actual implementation from t5.py =====
from __future__ import annotations

import math

import numpy as np

SEED = 0

PAD, BOS, EOS = 0, 1, 2

N_SENTINEL = 2

SENTINEL0 = 3

def sinusoidal_encoding(L: int, d_model: int) -> np.ndarray:
    """PE[pos,2i]=sin(pos/10000^{2i/d}); PE[pos,2i+1]=cos(...). See transformer.py."""
    pos = np.arange(L)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000.0, (2 * (i // 2)) / d_model)
    pe = np.zeros((L, d_model))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 64):
        super().__init__()
        pe = torch.tensor(sinusoidal_encoding(max_len, d_model), dtype=torch.float32)
        self.register_buffer("pe", pe)

    def forward(self, x):                       # x: (B, L, d)
        return x + self.pe[: x.size(1)]

class MultiHeadAttention(nn.Module):
    """Generic MHA usable for self- and cross-attention; additive mask."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def _split(self, x):
        B, L, _ = x.shape
        return x.view(B, L, self.h, self.d_k).transpose(1, 2)

    def forward(self, q, k, v, mask=None):      # mask: additive (..., Lq, Lk)
        Q, K, V = self._split(self.wq(q)), self._split(self.wk(k)), self._split(self.wv(v))
        scores = Q @ K.transpose(-1, -2) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores + mask
        attn = self.drop(scores.softmax(-1))
        o = (attn @ V).transpose(1, 2).reshape(q.size(0), -1, self.h * self.d_k)
        return self.out(o)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(),
                                 nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x): return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):        # pre-norm + residual
        x = x + self.attn(self.n1(x), self.n1(x), self.n1(x), src_mask)
        x = x + self.ff(self.n2(x))
        return x

def causal_mask(L: int, device) -> torch.Tensor:
    """Additive (1, 1, L, L) mask: 0 on/below diagonal, -inf above."""
    m = torch.triu(torch.ones(L, L, device=device), diagonal=1).bool()
    return torch.zeros(L, L, device=device).masked_fill(m, float("-inf"))[None, None]

def make_reverse_data(n: int, L: int, vocab: int, seed: int = SEED):
    """src = random content tokens; tgt = reversed, framed with BOS/EOS.
    A clean instance of the text-to-text view: input string -> output string."""
    rng = np.random.default_rng(seed)
    lo = SENTINEL0 + N_SENTINEL                              # content tokens above sentinels
    src = rng.integers(lo, vocab, size=(n, L))
    rev = src[:, ::-1]
    bos = np.full((n, 1), BOS)
    eos = np.full((n, 1), EOS)
    tgt_in = np.concatenate([bos, rev], axis=1)             # teacher-forcing input
    tgt_out = np.concatenate([rev, eos], axis=1)            # shifted target
    return src.astype(np.int64), tgt_in.astype(np.int64), tgt_out.astype(np.int64)

def demo():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    torch.set_num_threads(1)        # tiny CPU model: 1 thread avoids oversubscription
    dev = get_device()

    # First: illustrate the span-corruption objective on one toy sentence.
    rng = np.random.default_rng(SEED)
    sentence = [10, 11, 12, 13, 14, 15]
    src_sc, tgt_sc = span_corrupt(sentence, rng, p=0.5)
    print("span corruption (objective):")
    print("  original :", sentence)
    print("  input    :", src_sc, "  (sentinels start at id %d)" % SENTINEL0)
    print("  target   :", tgt_sc)

    # Then: train the seq2seq model on the text-to-text REVERSE task.
    V, L = 20, 6
    src, tin, tout = make_reverse_data(512, L, V)
    src = torch.tensor(src, device=dev)
    tin = torch.tensor(tin, device=dev)
    tout = torch.tensor(tout, device=dev)

    model = T5(V, d_model=48, n_heads=4, d_ff=96, n_layers=2,
               max_len=L + 2).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    lossfn = nn.CrossEntropyLoss(ignore_index=PAD)

    model.train()
    tmask = causal_mask(tin.size(1), dev)
    print("\ntext-to-text REVERSE task:")
    for step in range(1, 301):
        logits = model(src, tin, tgt_mask=tmask)
        loss = lossfn(logits.reshape(-1, V), tout.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 75 == 0:
            print(f"  step {step:4d}  loss {loss.item():.4f}")

    # greedy decode one example
    model.eval()
    with torch.no_grad():
        s = src[:1]
        mem = model.encode(s)
        ys = torch.tensor([[BOS]], device=dev)
        for _ in range(L):
            m = causal_mask(ys.size(1), dev)
            logit = model.decode(ys, mem, tgt_mask=m)
            nxt = logit[:, -1].argmax(-1, keepdim=True)
            ys = torch.cat([ys, nxt], dim=1)
    print("\n  src      :", s[0].tolist())
    print("  reversed :", s[0].flip(0).tolist())
    print("  predicted:", ys[0, 1:].tolist())


def span_corrupt(tokens: list[int], rng, p: float = 0.4,
                 sentinel0: int = SENTINEL0):
    r"""
    Span corruption (the T5 denoising objective), single sequence.

    Mark ~p of tokens for corruption; merge consecutive marked tokens into spans.
    Each span is replaced *in the input* by a unique sentinel <extra_id_k>, and the
    *target* is the concatenation of sentinels followed by their dropped spans:

        input  :  the <X> walked <Y> dog
        target :  <X> cat <Y> the    (then EOS)

    So the encoder sees a corrupted string and the decoder reconstructs only the
    missing pieces (much cheaper than reproducing the whole input).
    Returns (corrupted_input, target).
    """
    mark = rng.random(len(tokens)) < p
    src, tgt = [], []
    k = 0
    i = 0
    n = len(tokens)
    while i < n:
        if mark[i]:
            sent = sentinel0 + k
            src.append(sent)            # one sentinel for the whole span
            tgt.append(sent)
            while i < n and mark[i]:     # collect the contiguous span
                tgt.append(tokens[i])
                i += 1
            k += 1
        else:
            src.append(tokens[i])
            i += 1
    return src, tgt


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.n1 = nn.LayerNorm(d_model)
        self.n2 = nn.LayerNorm(d_model)
        self.n3 = nn.LayerNorm(d_model)

    def forward(self, x, mem, tgt_mask=None, src_mask=None):
        x = x + self.self_attn(self.n1(x), self.n1(x), self.n1(x), tgt_mask)
        nx = self.n2(x)
        x = x + self.cross_attn(nx, mem, mem, src_mask)         # attend to encoder
        x = x + self.ff(self.n3(x))
        return x

## 5. Full model — T5 encoder-decoder (shared/tied embedding)

In [ ]:
# ===== actual implementation from t5.py =====
class T5(nn.Module):
    """Encoder-decoder text-to-text Transformer (shared vocab + tied embedding)."""

    def __init__(self, vocab: int, d_model: int = 64, n_heads: int = 4,
                 d_ff: int = 128, n_layers: int = 2, max_len: int = 64,
                 dropout: float = 0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model, padding_idx=PAD)  # shared enc/dec
        self.pos = PositionalEncoding(d_model, max_len)
        self.enc = nn.ModuleList(EncoderLayer(d_model, n_heads, d_ff, dropout)
                                 for _ in range(n_layers))
        self.dec = nn.ModuleList(DecoderLayer(d_model, n_heads, d_ff, dropout)
                                 for _ in range(n_layers))
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.emb.weight                       # weight tying
        self.d_model = d_model

    def encode(self, src, src_mask=None):
        x = self.pos(self.emb(src) * math.sqrt(self.d_model))
        for layer in self.enc:
            x = layer(x, src_mask)
        return x

    def decode(self, tgt, mem, tgt_mask=None, src_mask=None):
        x = self.pos(self.emb(tgt) * math.sqrt(self.d_model))
        for layer in self.dec:
            x = layer(x, mem, tgt_mask, src_mask)
        return self.head(x)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        return self.decode(tgt, self.encode(src, src_mask), tgt_mask, src_mask)

## 6. Train / run — span-corruption demo + a text-to-text REVERSE task

In [ ]:
demo()

## 7. Visualization — span corruption + the seq2seq training loss

In [ ]:
import matplotlib
matplotlib.use("Agg")
import numpy as np, torch, matplotlib.pyplot as plt
import t5 as M

torch.manual_seed(M.SEED); np.random.seed(M.SEED); torch.set_num_threads(1)

# (a) train the REVERSE task and record the loss curve
V, L = 20, 6
src, tin, tout = M.make_reverse_data(512, L, V)
src = torch.tensor(src); tin = torch.tensor(tin); tout = torch.tensor(tout)
model = M.T5(V, d_model=48, n_heads=4, d_ff=96, n_layers=2, max_len=L + 2)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
lossfn = torch.nn.CrossEntropyLoss(ignore_index=M.PAD)
tmask = M.causal_mask(tin.size(1), torch.device("cpu"))
losses = []
for _ in range(150):
    logits = model(src, tin, tgt_mask=tmask)
    loss = lossfn(logits.reshape(-1, V), tout.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

# (b) the decoder causal mask used for teacher forcing
cm = M.causal_mask(L + 1, torch.device("cpu"))[0, 0].numpy()
cm = np.where(np.isneginf(cm), np.nan, 0.0) + np.tril(np.ones_like(cm))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses); ax[0].set_xlabel("step"); ax[0].set_ylabel("cross-entropy")
ax[0].set_title("Seq2seq REVERSE loss going down")
ax[1].imshow(cm, cmap="Greys", vmin=0, vmax=1)
ax[1].set_title("Decoder causal mask (teacher forcing)")
ax[1].set_xlabel("key pos j"); ax[1].set_ylabel("query pos i")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- T5 unifies NLP under one **text-to-text** interface and one cross-entropy loss;
  task identity lives in the input text (and, in real T5, a task prefix).
- **Span corruption** is the encoder-decoder analogue of MLM: drop spans, emit
  sentinels, regenerate — cheaper and denser than reconstructing the whole input.
- The encoder is bidirectional (understanding) and the decoder autoregressive
  (generation); **cross-attention** lets each output token query the full source.
- Pitfalls: mismatched sentinel ids between input and target; forgetting the
  decoder causal mask (leaks the target); encoder-decoder doubles the parameter
  count vs a decoder-only model of equal depth.
- Compare: **[GPT](gpt.ipynb)** (decoder-only) and **[BERT](bert.ipynb)**
  (encoder-only).